In [ ]:
# from pydub import AudioSegment

# # merge audio
# audio = AudioSegment.empty()

# for i in range(1, 11):
#     path = f"../data/audio/poem1_versuri/vers{i}.wav"
#     audio += AudioSegment.from_wav(path)

# # Audio(audio.raw_data, rate=audio.frame_rate)
# audio

In [ ]:
from pydub import AudioSegment

audio = AudioSegment.from_file("../data/raw-audio/thetower_01_yeats_128kb.mp3")

# remove the intro
audio = audio[14_500:]

audio

### stress marker

In [ ]:
# from adapt_eeg.utils.audio_stress_identifier import (
#     AnalysisParams,
#     detect_stressed_syllables,
# )

# params = AnalysisParams()

# stresses = detect_stressed_syllables(audio)

# stress_times = [stress.nucleus for stress in stresses]

In [ ]:
# ============================================================
# INSTALL IF NEEDED
# ============================================================
# pip install numpy pandas scipy matplotlib pydub praat-parselmouth


from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pydub import AudioSegment
from scipy.signal import find_peaks
import parselmouth

# ============================================================
# PARAMETERS — EASY TO TWEAK
# ============================================================

# ------------------------------------------------------------
# A. SYLLABLE NUCLEUS DETECTION
# ------------------------------------------------------------

# de Jong & Wempe use 50 Hz when calculating intensity.
# In Praat this controls the intensity analysis window.
intensity_floor_hz = 50.0

# Later de Jong/Wempe script:
# threshold = 99th percentile intensity - 25 dB
silence_threshold_db = -25.0

# Minimum acoustic prominence of an intensity peak.
# Start permissively because we don't want to lose weak,
# unstressed syllables before stress analysis.
min_nucleus_prominence_db = 2.0


# ------------------------------------------------------------
# B. VOICING CHECK
# ------------------------------------------------------------

# Broad range used ONLY to decide whether a candidate
# syllable nucleus has voiced speech around it.
voicing_pitch_floor_hz = 75.0
voicing_pitch_ceiling_hz = 500.0

# Pitch contour resolution for voicing check
voicing_time_step_s = 0.01

# Examine F0 around the intensity nucleus rather than relying
# on exactly one pitch frame.
voicing_half_window_s = 0.04

# At least this fraction of F0 frames around the nucleus
# must be voiced.
min_voiced_fraction = 0.50


# ------------------------------------------------------------
# C. PITCH USED FOR STRESS ANALYSIS
# ------------------------------------------------------------

# The poetry paper used 165–255 Hz for its female speaker.
# Change these if analysing a different speaker.
stress_pitch_floor_hz = 165.0
stress_pitch_ceiling_hz = 255.0

stress_pitch_time_step_s = 0.01

# Use a high percentile rather than the absolute maximum
# to reduce sensitivity to isolated F0 tracking errors.
pitch_percentile = 90


# ------------------------------------------------------------
# D. LOCAL PROMINENCE / STRESS
# ------------------------------------------------------------

# Compare each syllable against this many syllables
# on EACH side.
neighbor_radius = 2

# Initial engineering weights.
# The poetry paper says pitch is the strongest indicator,
# but DOES NOT provide numerical weights.
pitch_weight = 0.50
intensity_weight = 0.30
duration_weight = 0.20

# Candidate threshold on standardized combined score.
# This is NOT from either paper.
stress_score_threshold = 0.50

# Require both pitch and intensity to be locally elevated,
# reflecting the poetry paper's "co-occurring pitch and
# intensity peaks" criterion.
require_pitch_intensity_cooccurrence = True


# ============================================================
# INPUT
#
# Either:
#   "path/to/file.wav"
#
# OR:
#   a pydub.AudioSegment
# ============================================================

audio_input = audio

# Example:
# audio_input = AudioSegment.from_file("poem.wav")


# ============================================================
# 1. LOAD AUDIO
# ============================================================

if isinstance(audio_input, AudioSegment):
    audio = audio_input

elif isinstance(audio_input, (str, Path)):
    audio = AudioSegment.from_file(str(audio_input))

else:
    raise TypeError("audio_input must be a path or pydub.AudioSegment")


audio = audio.set_channels(1)

sample_rate = audio.frame_rate
duration_s = len(audio) / 1000.0


# ------------------------------------------------------------
# Convert pydub samples to floating point approximately
# in [-1, 1].
#
# Important:
# We DO NOT normalize every recording to its own maximum.
# That would destroy useful amplitude information.
# ------------------------------------------------------------

samples = np.asarray(audio.get_array_of_samples(), dtype=np.float64)

full_scale = float(1 << (8 * audio.sample_width - 1))

waveform = samples / full_scale


# ============================================================
# 2. CREATE PRAAT SOUND
# ============================================================

sound = parselmouth.Sound(waveform, sampling_frequency=sample_rate)


# ============================================================
# 3. INTENSITY CONTOUR
#
# No fixed 0.01-s time step here.
#
# time_step=None lets Praat choose its normal step based on
# the requested intensity analysis.
# ============================================================

intensity = sound.to_intensity(
    minimum_pitch=intensity_floor_hz, time_step=None, subtract_mean=True
)

intensity_times = intensity.xs()

intensity_db = intensity.values[0].astype(np.float64)


# ============================================================
# 4. ROBUST SILENCE / PEAK-HEIGHT THRESHOLD
#
# Instead of:
#
#     max(intensity) - 25
#
# use:
#
#     99th percentile - 25
#
# This is less sensitive to one loud click or transient.
# ============================================================

intensity_q99 = np.nanquantile(intensity_db, 0.99)

minimum_peak_height_db = intensity_q99 + silence_threshold_db


# ============================================================
# 5. DETECT CANDIDATE SYLLABLE NUCLEI
#
# IMPORTANT:
#
# There is NO 0.20-second minimum-distance rule anymore.
#
# scipy.signal.find_peaks() handles:
#
#   height     -> must be loud enough
#   prominence -> must rise sufficiently above surrounding
#                 intensity contour
#
# prominence is our Python operationalisation of the
# peak/dip logic.
# ============================================================

candidate_peaks, peak_properties = find_peaks(
    intensity_db, height=minimum_peak_height_db, prominence=min_nucleus_prominence_db
)


candidate_times = intensity_times[candidate_peaks]

candidate_intensity_db = intensity_db[candidate_peaks]

candidate_prominence_db = peak_properties["prominences"]


print(f"Initial acoustic peak candidates: " f"{len(candidate_peaks)}")


# ============================================================
# 6. BROAD F0 CONTOUR FOR VOICING
#
# This contour is NOT yet used to measure stress.
#
# Its only purpose is:
#
# "Does this intensity peak occur in voiced speech?"
# ============================================================

voicing_pitch = sound.to_pitch_ac(
    time_step=voicing_time_step_s,
    pitch_floor=voicing_pitch_floor_hz,
    pitch_ceiling=voicing_pitch_ceiling_hz,
)

voicing_pitch_times = voicing_pitch.xs()

voicing_f0 = voicing_pitch.selected_array["frequency"].astype(np.float64)

# Praat reports 0 for unvoiced frames.
voicing_f0[voicing_f0 <= 0] = np.nan


# ============================================================
# 7. VOICING FILTER
#
# Instead of testing one exact pitch frame, inspect a small
# region around each intensity nucleus.
# ============================================================

keep_voiced = []

voiced_fractions = []

for nucleus_time in candidate_times:

    mask = (voicing_pitch_times >= nucleus_time - voicing_half_window_s) & (
        voicing_pitch_times <= nucleus_time + voicing_half_window_s
    )

    local_f0 = voicing_f0[mask]

    if len(local_f0) == 0:

        voiced_fraction = 0.0

    else:

        voiced_fraction = np.mean(np.isfinite(local_f0))

    voiced_fractions.append(voiced_fraction)

    keep_voiced.append(voiced_fraction >= min_voiced_fraction)


keep_voiced = np.asarray(keep_voiced, dtype=bool)

voiced_fractions = np.asarray(voiced_fractions, dtype=float)


# ============================================================
# ACCEPTED SYLLABLE NUCLEI
# ============================================================

nucleus_indices = candidate_peaks[keep_voiced]

nucleus_times = candidate_times[keep_voiced]

nucleus_intensity_db = candidate_intensity_db[keep_voiced]

nucleus_prominence_db = candidate_prominence_db[keep_voiced]

nucleus_voiced_fraction = voiced_fractions[keep_voiced]


print(f"Accepted voiced syllable nuclei: " f"{len(nucleus_times)}")


if len(nucleus_times) == 0:
    raise RuntimeError(
        "No syllable nuclei survived detection. "
        "Try lowering min_nucleus_prominence_db or "
        "min_voiced_fraction."
    )


# ============================================================
# 8. ESTIMATE ACOUSTIC SYLLABLE BOUNDARIES
#
# Rather than midpoint boundaries, use intensity valleys.
#
# Find all local minima of the intensity contour.
# ============================================================

valley_indices, _ = find_peaks(-intensity_db)


# We need N+1 boundaries for N syllables.
boundary_indices = np.zeros(len(nucleus_indices) + 1, dtype=int)


# ------------------------------------------------------------
# Boundary before first nucleus:
# use nearest preceding local intensity minimum.
# ------------------------------------------------------------

before_first = valley_indices[valley_indices < nucleus_indices[0]]

if len(before_first):
    boundary_indices[0] = before_first[-1]
else:
    boundary_indices[0] = 0


# ------------------------------------------------------------
# Boundaries between adjacent nuclei:
# choose the LOWEST intensity valley between them.
# ------------------------------------------------------------

for i in range(len(nucleus_indices) - 1):

    left_peak = nucleus_indices[i]
    right_peak = nucleus_indices[i + 1]

    possible_valleys = valley_indices[
        (valley_indices > left_peak) & (valley_indices < right_peak)
    ]

    if len(possible_valleys):

        lowest = possible_valleys[np.argmin(intensity_db[possible_valleys])]

        boundary_indices[i + 1] = lowest

    else:

        # Fallback:
        # simply find the lowest sampled intensity value
        # between the two nuclei.

        segment = intensity_db[left_peak : right_peak + 1]

        relative_minimum = np.argmin(segment)

        boundary_indices[i + 1] = left_peak + relative_minimum


# ------------------------------------------------------------
# Boundary after last nucleus
# ------------------------------------------------------------

after_last = valley_indices[valley_indices > nucleus_indices[-1]]

if len(after_last):
    boundary_indices[-1] = after_last[0]
else:
    boundary_indices[-1] = len(intensity_db) - 1


boundary_times = intensity_times[boundary_indices]


# ============================================================
# 9. SYLLABLE INTERVALS
# ============================================================

syllable_starts = boundary_times[:-1]

syllable_ends = boundary_times[1:]

syllable_durations = syllable_ends - syllable_starts


# ============================================================
# 10. SECOND PITCH CONTOUR FOR STRESS
#
# Separate from the broad voicing contour.
#
# The default 165–255 Hz follows the poetry paper's female
# speaker. Do not blindly use this for another speaker.
# ============================================================

stress_pitch = sound.to_pitch_ac(
    time_step=stress_pitch_time_step_s,
    pitch_floor=stress_pitch_floor_hz,
    pitch_ceiling=stress_pitch_ceiling_hz,
)

stress_pitch_times = stress_pitch.xs()

stress_f0 = stress_pitch.selected_array["frequency"].astype(np.float64)

stress_f0[stress_f0 <= 0] = np.nan


# ============================================================
# 11. EXTRACT FEATURES FOR EACH ESTIMATED SYLLABLE
#
# Features:
#
#   pitch     -> high-percentile F0
#   intensity -> RMS amplitude in dBFS
#   duration  -> valley-to-valley interval
# ============================================================

syllable_pitch_hz = []

syllable_rms_db = []


for start_s, end_s in zip(syllable_starts, syllable_ends):

    # --------------------------------------------------------
    # PITCH
    # --------------------------------------------------------

    pitch_mask = (stress_pitch_times >= start_s) & (stress_pitch_times < end_s)

    local_pitch = stress_f0[pitch_mask]

    local_pitch = local_pitch[np.isfinite(local_pitch)]

    if len(local_pitch):

        pitch_value = np.percentile(local_pitch, pitch_percentile)

    else:

        pitch_value = np.nan

    syllable_pitch_hz.append(pitch_value)

    # --------------------------------------------------------
    # RMS INTENSITY
    #
    # Since waveform is scaled relative to digital full-scale,
    # this is dBFS rather than calibrated SPL.
    #
    # Relative differences are what matter for local
    # prominence.
    # --------------------------------------------------------

    start_sample = max(0, int(round(start_s * sample_rate)))

    end_sample = min(len(waveform), int(round(end_s * sample_rate)))

    segment = waveform[start_sample:end_sample]

    if len(segment):

        rms = np.sqrt(np.mean(segment**2))

        rms_db = 20 * np.log10(rms + 1e-12)

    else:

        rms_db = np.nan

    syllable_rms_db.append(rms_db)


syllable_pitch_hz = np.asarray(syllable_pitch_hz, dtype=float)

syllable_rms_db = np.asarray(syllable_rms_db, dtype=float)


# ============================================================
# 12. LOCAL PROMINENCE
#
# Stress is relative:
#
# current syllable
# minus
# median of nearby syllables.
# ============================================================


def local_prominence(values, radius=2):

    values = np.asarray(values, dtype=float)

    output = np.full(len(values), np.nan)

    for i in range(len(values)):

        left = max(0, i - radius)

        right = min(len(values), i + radius + 1)

        neighbours = np.concatenate([values[left:i], values[i + 1 : right]])

        neighbours = neighbours[np.isfinite(neighbours)]

        if np.isfinite(values[i]) and len(neighbours):

            output[i] = values[i] - np.median(neighbours)

    return output


pitch_prominence = local_prominence(syllable_pitch_hz, neighbor_radius)

intensity_prominence = local_prominence(syllable_rms_db, neighbor_radius)

duration_prominence = local_prominence(syllable_durations, neighbor_radius)


# ============================================================
# 13. ROBUST STANDARDIZATION
#
# Put pitch, intensity, and duration prominence onto
# comparable scales.
# ============================================================


def robust_zscore(values):

    values = np.asarray(values, dtype=float)

    output = np.full(len(values), np.nan)

    valid = np.isfinite(values)

    if not np.any(valid):
        return output

    median = np.nanmedian(values)

    mad = np.nanmedian(np.abs(values - median))

    scale = 1.4826 * mad

    if not np.isfinite(scale) or scale == 0:

        scale = np.nanstd(values)

    if not np.isfinite(scale) or scale == 0:

        scale = 1.0

    output[valid] = (values[valid] - median) / scale

    return output


pitch_z = robust_zscore(pitch_prominence)

intensity_z = robust_zscore(intensity_prominence)

duration_z = robust_zscore(duration_prominence)


# ============================================================
# 14. CONTINUOUS STRESS SCORE
#
# Missing values are NOT interpolated.
# Instead, available weights are renormalized.
# ============================================================

stress_scores = np.full(len(nucleus_times), np.nan)


base_weights = np.array([pitch_weight, intensity_weight, duration_weight])


for i in range(len(nucleus_times)):

    features = np.array([pitch_z[i], intensity_z[i], duration_z[i]])

    available = np.isfinite(features)

    if not np.any(available):
        continue

    current_weights = base_weights[available]

    current_weights = current_weights / current_weights.sum()

    stress_scores[i] = np.sum(features[available] * current_weights)


# ============================================================
# 15. OPTIONAL BINARY STRESS-CANDIDATE LABEL
#
# Keep the continuous score regardless.
# ============================================================

stress_candidates = stress_scores >= stress_score_threshold


if require_pitch_intensity_cooccurrence:

    stress_candidates &= (
        np.isfinite(pitch_prominence)
        & (pitch_prominence > 0)
        & np.isfinite(intensity_prominence)
        & (intensity_prominence > 0)
    )


# ============================================================
# 16. RESULTS TABLE
# ============================================================

results = pd.DataFrame(
    {
        "syllable": np.arange(1, len(nucleus_times) + 1),
        "start_s": syllable_starts,
        "nucleus_s": nucleus_times,
        "end_s": syllable_ends,
        "duration_s": syllable_durations,
        # Detection information
        "nucleus_intensity_db": nucleus_intensity_db,
        "nucleus_prominence_db": nucleus_prominence_db,
        "voiced_fraction": nucleus_voiced_fraction,
        # Stress features
        "pitch_hz": syllable_pitch_hz,
        "rms_dbfs": syllable_rms_db,
        "pitch_prominence": pitch_prominence,
        "intensity_prominence": intensity_prominence,
        "duration_prominence": duration_prominence,
        "pitch_z": pitch_z,
        "intensity_z": intensity_z,
        "duration_z": duration_z,
        "stress_score": stress_scores,
        "stress_candidate": stress_candidates,
    }
)


pd.set_option("display.max_rows", None)

pd.set_option("display.width", 200)




# ============================================================
# 17. STRESS TIMES / INTER-STRESS INTERVALS
# ============================================================

stress_times = results.loc[results["stress_candidate"], "nucleus_s"].to_numpy()




if len(stress_times) > 1:

    inter_stress_intervals = np.diff(stress_times)

    median_interval = np.median(inter_stress_intervals)

    print("\nInter-stress intervals:")

    print(inter_stress_intervals)

    print(f"\nMedian inter-stress interval: " f"{median_interval:.3f} s")

    print(f"Approximate stress frequency: " f"{1 / median_interval:.3f} Hz")



In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(16, 5))

# # Raw / unfiltered audio waveform
# plt.plot(time, waveform, linewidth=0.6, alpha=0.8, label="Raw audio")

# # Vertical lines at detected stress candidates
# for stress_time in stress_times:
#     plt.axvline(x=stress_time, linewidth=1, alpha=0.7)

# plt.xlabel("Time (s)")
# plt.ylabel("Amplitude")
# plt.title("Raw audio with detected stress candidates")
# plt.tight_layout()
# plt.show()

### visualize code



In [ ]:
import base64
import io
import json
import mimetypes
import uuid
from pathlib import Path

from IPython.display import HTML, display
from pydub import AudioSegment


def _audio_to_data_url(audio):
    if isinstance(audio, AudioSegment):
        buf = io.BytesIO()
        audio.export(buf, format="wav")
        audio_bytes = buf.getvalue()
        mime_type = "audio/wav"

    elif isinstance(audio, (str, Path)):
        path = Path(audio)

        if not path.exists():
            raise FileNotFoundError(path)

        audio_bytes = path.read_bytes()
        mime_type = mimetypes.guess_type(path.name)[0] or "audio/mpeg"

    else:
        raise TypeError("audio must be a pydub.AudioSegment or a file path")

    encoded = base64.b64encode(audio_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def waveform_player(
    audio,
    timestamps,
    window_seconds=20,
    height=160,
):
    """
    Parameters
    ----------
    audio:
        pydub.AudioSegment or path to file

    timestamps:
        [1.2, 3.4, 8.1]

        or

        [
            (1.2, "marker A"),
            (3.4, "marker B"),
        ]

    window_seconds:
        Total visible time.
        20 means roughly +/- 10 seconds around playback.
    """

    audio_url = _audio_to_data_url(audio)

    markers = []

    for item in timestamps:
        if isinstance(item, (tuple, list)):
            t, label = item
        else:
            t = item
            label = f"{float(t):.2f}s"

        markers.append(
            {
                "time": float(t),
                "label": str(label),
            }
        )

    markers_json = json.dumps(markers)

    uid = uuid.uuid4().hex

    waveform_id = f"waveform_{uid}"
    button_id = f"button_{uid}"
    time_id = f"time_{uid}"

    html = f"""
    <style>
        #{waveform_id} {{
            width: 100%;
        }}

        .controls-{uid} {{
            margin-top: 10px;
            display: flex;
            align-items: center;
            gap: 12px;
            font-family: sans-serif;
        }}

        #{time_id} {{
            font-family: monospace;
            font-size: 13px;
        }}
    </style>

    <div id="{waveform_id}"></div>

    <div class="controls-{uid}">
        <button id="{button_id}">
            ▶ Play
        </button>

        <span id="{time_id}">
            0.00 s
        </span>
    </div>

    <script type="module">

        import WaveSurfer from
        "https://cdn.jsdelivr.net/npm/wavesurfer.js@7/dist/wavesurfer.esm.js";

        import RegionsPlugin from
        "https://cdn.jsdelivr.net/npm/wavesurfer.js@7/dist/plugins/regions.esm.js";


        const markers = {markers_json};

        const waveformContainer =
            document.getElementById("{waveform_id}");

        const button =
            document.getElementById("{button_id}");

        const timeDisplay =
            document.getElementById("{time_id}");


        const regions = RegionsPlugin.create();


        const wavesurfer = WaveSurfer.create({{
            container: waveformContainer,

            url: "{audio_url}",

            height: {height},

            waveColor: "#aaa",
            progressColor: "#444",

            cursorColor: "#2563eb",
            cursorWidth: 2,

            normalize: true,

            autoScroll: true,
            autoCenter: true,

            minPxPerSec: 50,

            plugins: [regions],
        }});


        let duration = 0;


        function updateZoom() {{
            const width =
                waveformContainer.clientWidth;

            if (!width) return;

            const pxPerSecond =
                width / {float(window_seconds)};

            wavesurfer.zoom(pxPerSecond);
        }}


        function addMarkers() {{

            regions.clearRegions();

            markers.forEach(marker => {{

                if (
                    marker.time < 0 ||
                    marker.time > duration
                ) {{
                    return;
                }}

                /*
                 * Very narrow region.
                 *
                 * Width is 0.01 sec, which appears
                 * as a vertical red marker when zoomed.
                 */
                const region = regions.addRegion({{
                    start: marker.time,
                    end: Math.min(
                        marker.time + 0.01,
                        duration
                    ),

                    color: "rgba(239, 68, 68, 0.95)",

                    drag: false,
                    resize: false,
                }});


                /*
                 * Tooltip
                 */
                region.element.title =
                    `${{marker.label}} (${{marker.time.toFixed(2)}}s)`;


                /*
                 * Make narrow regions easier to see.
                 */
                region.element.style.minWidth = "2px";
                region.element.style.width = "2px";
                region.element.style.background =
                    "rgb(239, 68, 68)";
                region.element.style.zIndex = "10";
                region.element.style.cursor = "pointer";


                /*
                 * Click marker to seek.
                 */
                region.element.addEventListener(
                    "click",
                    event => {{

                        event.stopPropagation();

                        wavesurfer.setTime(
                            marker.time
                        );
                    }}
                );
            }});
        }}


        wavesurfer.on("ready", () => {{

            duration =
                wavesurfer.getDuration();

            updateZoom();

            requestAnimationFrame(() => {{
                addMarkers();
            }});

            timeDisplay.textContent =
                `0.00 s / ${{duration.toFixed(2)}} s`;
        }});


        wavesurfer.on("timeupdate", time => {{

            timeDisplay.textContent =
                `${{time.toFixed(2)}} s / ` +
                `${{duration.toFixed(2)}} s`;
        }});


        wavesurfer.on("play", () => {{
            button.textContent = "⏸ Pause";
        }});


        wavesurfer.on("pause", () => {{
            button.textContent = "▶ Play";
        }});


        wavesurfer.on("finish", () => {{
            button.textContent = "▶ Play";
        }});


        button.addEventListener("click", () => {{
            wavesurfer.playPause();
        }});


        const resizeObserver =
            new ResizeObserver(() => {{

                if (!duration) return;

                const currentTime =
                    wavesurfer.getCurrentTime();

                updateZoom();

                requestAnimationFrame(() => {{
                    addMarkers();

                    wavesurfer.setTime(
                        currentTime
                    );
                }});
            }});


        resizeObserver.observe(
            waveformContainer
        );

    </script>
    """

    display(HTML(html))

### visualize

In [ ]:
waveform_player(audio, stress_times)